# Tool use — custom (client) tools

Custom tools are the core agentic primitive: you declare tools, Claude decides when to call one,
**you** execute it, and you feed the result back — looping until Claude is done. This is the
opposite of the `code_execution/` *server* tool (where Anthropic runs the code); here you control
the implementations.

We define two tiny tools — a safe `calculator` and a mock `get_weather` — and use them three
ways: a single call, a manual agentic loop, and the SDK's automatic tool runner.

**Requirements:** `ANTHROPIC_API_KEY` (env var or a `.env` at the repo root).

## Setup

The tool implementations, JSON schemas, dispatch, and a manual loop live in `_tool_use.py`.

In [ ]:
import os
import sys

from dotenv import load_dotenv
from anthropic import Anthropic

for _p in (".", "tool_use"):
    if os.path.isfile(os.path.join(_p, "_tool_use.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _tool_use import MODEL, TOOLS, execute_tool, run_loop

load_dotenv()
client = Anthropic()

## The tools

A tool is a name, a description, and a JSON-schema for its inputs. Claude never runs your code —
it emits a `tool_use` block requesting a call; you run it and return a `tool_result`.

In [ ]:
TOOLS

## One step, by hand

Ask something that needs a tool. The response `stop_reason` is `tool_use`, and the content holds
a `tool_use` block with the arguments Claude chose. You execute it and send back a `tool_result`
(matched by `tool_use_id`).

In [ ]:
resp = client.messages.create(
    model=MODEL, max_tokens=1024, tools=TOOLS,
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
)
print("stop_reason:", resp.stop_reason)
for block in resp.content:
    if block.type == "tool_use":
        print("tool_use:", block.name, block.input)
        content, is_error = execute_tool(block.name, block.input)
        print("result:", content)

## The agentic loop

`run_loop` repeats that exchange until Claude stops calling tools — handling multiple/parallel
tool calls per turn. It returns the final text and a transcript of every step.

In [ ]:
final, transcript = run_loop(
    client, "What's the weather in Paris, and what is 18°C in Fahrenheit? Use the tools.")

for e in transcript:
    if e["kind"] == "text":
        print("ASSISTANT:", e["text"])
    elif e["kind"] == "tool_call":
        print(f"  → call {e['name']}({e['input']})")
    elif e["kind"] == "tool_result":
        print(f"  ← {e['content']}" + ("  [error]" if e["is_error"] else ""))

print("\nFINAL:\n" + final)

## The easy way: the tool runner

The SDK can run the whole loop for you. Decorate typed functions with `@beta_tool` (the schema is
generated from the signature + docstring) and pass them to `client.beta.messages.tool_runner` —
no manual loop, no result plumbing.

In [ ]:
from anthropic import beta_tool
from _tool_use import calculator as _calc, get_weather as _wx

@beta_tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression.

    Args:
        expression: An arithmetic expression, e.g. '100 - 32'.
    """
    return _calc(expression)

@beta_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: City name, e.g. 'Sydney'.
    """
    return _wx(city)

runner = client.beta.messages.tool_runner(
    model=MODEL, max_tokens=1024, tools=[calculator, get_weather],
    messages=[{"role": "user", "content": "What's the weather in Sydney, and what is 100 - 32?"}],
)
for message in runner:
    for block in message.content:
        if block.type == "text" and block.text.strip():
            print(block.text)

## `tool_choice` and error handling

Force a specific tool with `tool_choice`. And when a tool fails, return the error as a
`tool_result` with `is_error: True` — Claude reads it and recovers (e.g. asks for a valid city)
rather than crashing.

In [ ]:
# Force the calculator
forced = client.messages.create(
    model=MODEL, max_tokens=512, tools=TOOLS,
    tool_choice={"type": "tool", "name": "calculator"},
    messages=[{"role": "user", "content": "Use a tool to compute 2**10."}],
)
print("forced tool:", [b.name for b in forced.content if b.type == "tool_use"])

# Error path: an unknown city returns is_error=True
print("error result:", execute_tool("get_weather", {"city": "Atlantis"}))

## Notes

- **Client vs server tools.** Here *you* execute tools and return `tool_result`s. Contrast
  `code_execution/`, a *server* tool Anthropic runs with no loop for you to write.
- **Parallel tools.** Claude may emit several `tool_use` blocks in one turn — execute them all and
  return all the `tool_result`s in a single user message.
- **`is_error`.** Return failures as tool results with `is_error: True` instead of raising, so
  Claude can adapt.
- **Manual loop vs runner.** Use the runner for speed; use the manual loop when you need
  control — approval gates, logging, conditional execution, custom transcripts (as here).
- **Structured tool inputs.** For guaranteed-valid arguments, add `"strict": True` to a tool's
  schema (see the `structured_outputs/` topic).
- **MCP.** The same runner can drive tools from an MCP server via the SDK's MCP helpers.